# MK-UNet on Google ColabTraining notebook for the MK-UNet tumor-segmentation project (Phase 1 baseline).**Runtime → Change runtime type → T4 GPU → Save**, then **Connect**, then run cells 1–8 top to bottom.## One-time setup (do this once, ever)**Order matters here** — `My Drive/mkunet/data/` does not exist yet. Cell 4 creates it.1. **Fork the repo.** Go to <https://github.com/SLDGroup/MK-UNet> and click **Fork**. You need a   repo you can push to, since Phase 3 involves modifying the architecture.2. **Set `GITHUB_USER` in cell 1** to your GitHub username.3. **Run cells 1 through 4.** Cell 4 mounts Drive and creates the `mkunet/` folder tree,   including `mkunet/data/`.4. **Now add the dataset.** Open the folder link below in a browser tab, right-click the folder →   **Organize → Add shortcut to Drive** → choose `My Drive/mkunet/data/`.   A shortcut is a pointer, not a copy, so it uses none of your Drive quota.   - ClinicDB: <https://drive.google.com/drive/folders/1FPJr5f91uUCikxMvkwtZSEnYHemTZq1P>   - ColonDB: <https://drive.google.com/drive/folders/1u4_8dMztnEBUaX-w3XfUR3jXLBhpccPA>5. **Run cells 5 onward.**If you would rather not wait: the Add-shortcut dialog has a **New folder** button, so you cancreate `mkunet/data` there directly and run everything straight through. Cell 6 also accepts theshortcut sitting at the top level of My Drive (`My Drive/ClinicDB`) — it checks several locationsand prints every path it tried if it comes up empty.## Every session after thatJust run cells 1–8. The Colab machine is destroyed when your session ends, so packages must bereinstalled and the repo re-cloned each time — that is normal, not a failure. Anything that mustsurvive lives in Google Drive.

## 1 · ConfigurationEverything you would normally tweak lives here. Nothing below this cell needs editing.

In [ ]:
# ---- your fork -------------------------------------------------------------GITHUB_USER = "EliBaumgardner"      # <<< CHANGE ME to your GitHub usernameREPO        = "MK-UNet"# ---- experiment ------------------------------------------------------------DATASET   = "ClinicDB"    # "ClinicDB" or "ColonDB"NETWORK   = "MK_UNet"     # MK_UNet_T | MK_UNet_S | MK_UNet | MK_UNet_M | MK_UNet_LNUM_RUNS  = 1             # 1 to verify the pipeline; 5 for publishable mean +/- stdEPOCHS    = 200BATCHSIZE = 8IMG_SIZE  = 352LR        = 0.0005# ---- paths (leave alone) ---------------------------------------------------DRIVE_ROOT = "/content/drive/MyDrive/mkunet"REPO_DIR   = f"/content/{REPO}"DATA_DST   = f"{REPO_DIR}/data/polyp/target"DATASET_FOLDER_IDS = {    "ClinicDB": "1FPJr5f91uUCikxMvkwtZSEnYHemTZq1P",    "ColonDB":  "1u4_8dMztnEBUaX-w3XfUR3jXLBhpccPA",}print(f"{NETWORK} on {DATASET} | {NUM_RUNS} run(s) x {EPOCHS} epochs | batch {BATCHSIZE}")if NUM_RUNS == 1:    print("NOTE: 1 run gives a single number with unknown seed luck. Use 5 for paper results.")

## 2 · Confirm you actually got a GPUIf this says `cuda available: False`, you are on a CPU runtime and training will be unusably slow. Fix it under Runtime → Change runtime type.

In [ ]:
import torch, subprocessprint(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],                     capture_output=True, text=True).stdout.strip() or "NO GPU DETECTED")print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Connect."

## 3 · Mount Google DriveThis pops an authorization dialog — click through it. Drive is where checkpoints, logs and results are kept, because the Colab machine itself is temporary.

In [ ]:
import osfrom google.colab import drivedrive.mount("/content/drive")for sub in ["model_pth", "logs", "results_polyp", "predictions_polyp", "data"]:    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)print("Drive ready at", DRIVE_ROOT)

## 4 · Get the codeClones your fork, or pulls the latest if it is already there. This is the bridge between your Mac and Colab: edit in PyCharm → `git push` → re-run this cell → your changes are live.

In [ ]:
import os, subprocessdef sh(cmd, cwd=None):    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)    print((r.stdout + r.stderr).strip())    return r.returncodeif os.path.isdir(f"{REPO_DIR}/.git"):    sh("git pull", cwd=REPO_DIR)else:    if sh(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}") != 0:        print(f"\nFork not found. Falling back to upstream (you will NOT be able to push).")        sh(f"git clone https://github.com/SLDGroup/{REPO}.git {REPO_DIR}")os.chdir(REPO_DIR)print("\ncwd:", os.getcwd())

## 5 · Install dependenciesOnly six packages are missing from Colab's defaults.**Do not follow the repo README's environment section.** It specifies `torch==1.11.0+cu113` and`mmcv-full`, which will not install on Colab's Python 3.12 / CUDA 12.x. `mmcv` is also neverimported anywhere in this repo — nor are `transformers`, `einops`, `ml_collections`,`warmup-scheduler`, `ptflops`, `torchprofile`, `huggingface-hub`, `tensorboardx`, `torchmetrics`,`nibabel`, `h5py` or `tifffile`. `requirements.txt` is inherited from EMCAD/CASCADE and is mostlydead weight.`timm` is pinned to 0.9.16 because `mkunet_network.py` lines 9–10 import from the deprecated`timm.models.layers` / `timm.models.helpers` shim paths, which newer releases have been removing.

In [ ]:
%pip install -q timm==0.9.16 albumentations medpy SimpleITK segmentation-mask-overlay thop openpyxlprint("done - ignore any pip dependency-resolver warnings about unrelated Colab packages")

## 6 · Stage the datasetLooks for the dataset in Drive (from the shortcut you made in the one-time setup) and copies it tothe Colab machine's **local disk**. Training reads thousands of small image files, and local diskis far faster than the Drive network mount.**This is the cell that fails if you skipped the shortcut step.** It searches four locations andprints all of them on failure, so read the error rather than guessing. `My Drive/mkunet/data/ClinicDB`and `My Drive/ClinicDB` both work.The `gdown` fallback is a courtesy, not a solution: it caps folder downloads at 50 files and willtruncate these datasets. Use the shortcut.

In [ ]:
import os, glob, shutil, subprocessos.makedirs(DATA_DST, exist_ok=True)dst = f"{DATA_DST}/{DATASET}"def looks_valid(p):    return os.path.isdir(os.path.join(p, "train", "images"))if looks_valid(dst):    print("Already staged on local disk:", dst)else:    candidates = [        f"{DRIVE_ROOT}/data/{DATASET}",        f"/content/drive/MyDrive/{DATASET}",        f"{DRIVE_ROOT}/data/target/{DATASET}",        f"/content/drive/MyDrive/data/{DATASET}",    ]    src = next((c for c in candidates if looks_valid(c)), None)    if src:        print(f"Copying {src} -> {dst} ...")        subprocess.run(f'cp -rL "{src}" "{dst}"', shell=True, check=True)        print("Staged from Drive.")    else:        print("Not found in Drive. Falling back to direct download...")        print("Checked:", *candidates, sep="\n  ")        subprocess.run("pip install -q --upgrade gdown", shell=True)        fid = DATASET_FOLDER_IDS[DATASET]        subprocess.run(            f'gdown --folder "https://drive.google.com/drive/folders/{fid}" '            f'-O "{DATA_DST}" --remaining-ok', shell=True)        if not looks_valid(dst):            raise SystemExit(                "\nDataset still missing.\n"                "gdown caps folder downloads at 50 files, so it often truncates this dataset.\n"                "Do the one-time Drive step instead:\n"                f"  1. Open https://drive.google.com/drive/folders/{fid}\n"                "  2. Right-click the folder -> Organize -> Add shortcut to Drive\n"                "  3. Place it in  My Drive/mkunet/data/\n"                "  4. Re-run this cell.")# report what we actually haveprint()for split in ["train", "val", "test"]:    for kind in ["images", "masks"]:        p = os.path.join(dst, split, kind)        n = len(glob.glob(os.path.join(p, "*"))) if os.path.isdir(p) else -1        print(f"  {split:<5} {kind:<7} {n if n >= 0 else 'MISSING':>6}")

## 7 · Point outputs at DriveCheckpoints, logs and results are written to folders inside the repo. These symlinks redirect them into Drive so they survive the session ending.

In [ ]:
import os, subprocessfor name in ["model_pth", "logs", "results_polyp", "predictions_polyp"]:    local, remote = f"{REPO_DIR}/{name}", f"{DRIVE_ROOT}/{name}"    os.makedirs(remote, exist_ok=True)    if os.path.islink(local):        continue    if os.path.isdir(local):        subprocess.run(f'rm -rf "{local}"', shell=True)    os.symlink(remote, local)    print(f"{name} -> {remote}")print("\nOutputs will persist in Drive.")

## 8 · Sanity check + apply configVerifies every import resolves and the model builds, then patches `train_polyp.py` for your chosen dataset and run count. This is cheap insurance against discovering a broken import forty minutes into training.

In [ ]:
import os, re, torchos.chdir(REPO_DIR)# --- imports + forward pass ---from mkunet_network import MK_UNetfrom utils.dataloader_polyp import get_loaderimport timm, albumentations, medpy, SimpleITK, thop  # noqa: F401NET_CONFIGS = {'MK_UNet_T':[4,8,16,24,32], 'MK_UNet_S':[8,16,32,48,80],               'MK_UNet':[16,32,64,96,160], 'MK_UNet_M':[32,64,128,192,320],               'MK_UNet_L':[64,128,256,384,512]}m = MK_UNet(num_classes=1, in_channels=3, channels=NET_CONFIGS[NETWORK]).cuda()out = m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).cuda())out = out[0] if isinstance(out, (list, tuple)) else outprint(f"{NETWORK}: {sum(p.numel() for p in m.parameters())/1e6:.3f}M params | output {tuple(out.shape)}")del m, out; torch.cuda.empty_cache()# --- patch train_polyp.py (idempotent) ---p = f"{REPO_DIR}/train_polyp.py"s = open(p).read()s = re.sub(r"for run in \[[^\]]*\]:", f"for run in {list(range(1, NUM_RUNS+1))}:", s, count=1)s = re.sub(r"dataset_name = '[^']*'", f"dataset_name = '{DATASET}'", s, count=1)open(p, "w").write(s)for line in open(p):    if line.strip().startswith(("for run in", "dataset_name =")):        print("patched:", line.rstrip())

## 9 · Smoke test (2 epochs)Run this **before** committing to a full training job. It confirms the data loads, the loss goesdown and checkpoints get written. Watch the `Step [xxxx/xxxx]` counter — the second number is yourbatches per epoch, which is what you multiply to estimate total runtime.

In [ ]:
cmd = (f"cd {REPO_DIR} && python -W ignore train_polyp.py --network {NETWORK} "       f"--epoch 2 --batchsize {BATCHSIZE} --img_size {IMG_SIZE} --lr {LR}")print(cmd, "\n")!{cmd}

## 10 · Full training**Before starting:** enable background execution so the job survives closing the tab —this is a Colab Pro feature and the main reason Pro is worth it here.Note that each batch is trained at three scales (0.75x, 1.0x, 1.25x — `train_polyp.py` line 112),so one epoch costs roughly three times what the batch count alone suggests. Validation and testare also evaluated every epoch.

In [ ]:
cmd = (f"cd {REPO_DIR} && python -W ignore train_polyp.py --network {NETWORK} "       f"--epoch {EPOCHS} --batchsize {BATCHSIZE} --img_size {IMG_SIZE} --lr {LR}")print(cmd, "\n")!{cmd}

## 11 · EvaluateFinds the most recent run automatically. `run_id` embeds a timestamp, so it cannot be reconstructed by hand.

In [ ]:
import glob, osruns = sorted(glob.glob(f"{REPO_DIR}/model_pth/*"), key=os.path.getmtime)assert runs, "No trained runs found in model_pth/."RUN_ID = os.path.basename(runs[-1])print("Testing:", RUN_ID)print("Checkpoints:", *[os.path.basename(f) for f in glob.glob(f"{runs[-1]}/*.pth")])

In [ ]:
cmd = (f"cd {REPO_DIR} && python -W ignore test_polyp.py --network {NETWORK} "       f"--run_id {RUN_ID} --dataset_name {DATASET}")print(cmd, "\n")!{cmd}

## 12 · ResultsThese numbers are your Phase 1 baseline. Record them in ProjectFlow.md — every later change gets measured against them.

In [ ]:
import glob, pandas as pdfor f in sorted(glob.glob(f"{REPO_DIR}/results_polyp/*.xlsx")):    print("\n==", f.split("/")[-1], "==")    display(pd.read_excel(f))

---## Troubleshooting| Symptom | Cause | Fix ||---|---|---|| `No GPU` assertion in cell 2 | CPU runtime | Runtime → Change runtime type → T4 GPU || `FileNotFoundError` on `images/` | Dataset not staged | Re-run cell 6; do the Drive shortcut step || Your code change had no effect | Colab has a stale clone | `git push` on the Mac, re-run cell 4 || `ImportError` from `timm` | Wrong timm version | Re-run cell 5; the 0.9.16 pin matters || Everything vanished | Session was recycled | Expected. Re-run cells 1–8. Drive contents are safe. || Training died partway | Session limit or disconnect | Checkpoints are in Drive, but this repo has **no resume logic** — it restarts from scratch |## Things worth knowing about this codebase- `dataset_name` is hardcoded at `train_polyp.py:165`, not a CLI flag. Cell 8 patches it.- `for run in [1,2,3,4,5]` at `train_polyp.py:201` means five full trainings in one process.  Cell 8 patches it to `NUM_RUNS`.- `size_rates = [0.75, 1, 1.25]` at `train_polyp.py:112` triples per-epoch cost.- The best checkpoint is selected on **validation** Dice, and the test score at that epoch is  reported separately (`train_polyp.py:155-161`) — this is the correct protocol, not a bug.